In [1]:
from spin_lattices import KagomeLattice, SpinLattice, ChainLattice, SquareLattice
from heisenberg_hamiltonians import HeisenbergJ1J2, SpinSystem
from pathlib import Path
import networkx as nx
import numpy as np
from typing import Callable
import torch
import numpy.typing as npt
import lattice_symmetries as ls
from typing import Any, Optional, Union, Dict, Tuple
from loguru import logger
from collections import namedtuple
from torch import Tensor
import torch.nn as nn
from misc_utils import make_unpacked_configurations
import io
from contextlib import redirect_stderr
from torch.utils.tensorboard import SummaryWriter
from datetime import datetime
from nqs_playground_helpers import (
    SamplingOptions,
    split_into_batches,
    safe_exp,
    sample_exactly,
    sample_full,
    forward_with_batches,
)
from scipy.sparse import csr_matrix, coo_matrix, diags
from scipy.sparse.csgraph import connected_components
import sys
from kagome_cnn import KagomeCNNRegression
from torch.nn.utils import parameters_to_vector


2023-08-01 19:37:21.558 | DEBUG    | lattice_symmetries:__init__:50 - Initializing Haskell runtime...
2023-08-01 19:37:21.580 | DEBUG    | lattice_symmetries:__init__:52 - Initializing Chapel runtime...
2023-08-01 19:37:21.941 | DEBUG    | lattice_symmetries:__init__:54 - Setting Python exception handler...
set_python_exception_handler ...


In [2]:
logger.remove()
logger.add(sys.stderr, level="INFO")

def find_overlap(x, y):
    x = x.view(-1)
    y = y.view(-1)
    return torch.sum(x * y) / torch.sqrt(torch.sum(x**2) * torch.sum(y**2))

In [3]:
# lattice = ChainLattice(10)
# system = HeisenbergJ1J2(
#     lattice=lattice,
#     J1=1,
#     J2=1,
#     ground_state_cache_dir=Path("groundstates"),
#     use_symmetries=False,
#     spin_inversion=None,
# )
# system.hamiltonian.apply_off_diag_to_basis_state(system.hamiltonian.basis.states[0])

In [4]:
class LogProbDenseNet(nn.Module):
    def __init__(self, system: SpinSystem, n_hidden: int = 100, hidden_layers=1):
        super().__init__()
        self.system = system
        self.n_hidden = n_hidden
        self.hidden_layers = hidden_layers
        layers = [nn.Linear(system.number_spins, n_hidden), nn.ReLU()]
        for _ in range(hidden_layers - 1):
            layers.append(nn.Linear(n_hidden, n_hidden))
            layers.append(nn.ReLU())

        layers.append(nn.Linear(n_hidden, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x: Tensor) -> Tensor:
        return self.net(
            torch.from_numpy(
                make_unpacked_configurations(x, self.system.number_spins).astype(np.float32)
            )
        )

In [5]:
def find_nbd(
    hamiltonian: ls.Operator, states: npt.NDArray[np.uint64]
) -> tuple[csr_matrix, npt.NDArray[np.uint64]]:
    """
    Constructs a sparse matrix that is a slice of the Hamiltonian matrix. 
    Included rows are determined by ``states``, included columns 

    Parameters
    ----------
    system : SpinSystem
        The system to construct the matrix for.

    states : npt.NDArray[np.uint64]
        The states whose neighbors to include in the matrix.

    Returns
    -------
    M : csr_matrix
        The sparse matrix.
    
    nbd_states : npt.NDArray[np.uint64]
        The sorted array of neighbors.
        
        The following holds:

        ``M[i, j] = <states[i] | H | nbd_states[j]>``
    """
    coeff_rows = []
    nbd_states_rows = []
    row_indices = [0]
    for state in states:
        # process neighbors
        cur_coeffs, cur_nbd_states = map(
            list, zip(*hamiltonian.apply_off_diag_to_basis_state(state))
        )

        # process self
        cur_coeffs.append(hamiltonian.apply_diag_to_basis_state(state))
        cur_nbd_states.append(state)

        # make rows
        coeff_rows.append(cur_coeffs)
        nbd_states_rows.append(cur_nbd_states)
        row_indices.append(row_indices[-1] + len(cur_coeffs))

    coeffs_data = np.concatenate(coeff_rows).astype(np.float64)
    nbd_states_data = np.concatenate(nbd_states_rows)
    nbd_states = np.unique(nbd_states_data)
    nbd_indices = np.searchsorted(nbd_states, nbd_states_data)
    row_indices = np.array(row_indices)

    return (
        csr_matrix((coeffs_data, nbd_indices, row_indices), shape=(len(states), len(nbd_states))),
        nbd_states,
    )

In [6]:
# def nbd_matrix_to_graph(
#     states: npt.NDArray, nbd_matrix: csr_matrix, nbd_states: npt.NDArray
# ) -> csr_matrix:
#     """
#     Constructs a graph from a neighborhood matrix (see ``find_nbd``):

#     - Expands matrix to make it square. Rows are rearranged to align them
#         with columns, indexed by ``nbd_states``. I.e. row ``i`` corresponds to
#         ``nbd_states[i]``.

#     - Symmetrizes the matrix.

#     - Converts to a graph by thresholding at 0.
#     """
#     symmetric_matrix = csr_matrix((len(nbd_states), len(nbd_states)))
#     state_indices = np.searchsorted(nbd_states, states)
#     symmetric_matrix[state_indices, :] = (nbd_matrix != 0).astype(np.uint8)
#     symmetric_matrix += symmetric_matrix.T
#     return (symmetric_matrix != 0).astype(np.uint8)


def nbd_matrix_to_graph(
    states: np.ndarray, nbd_matrix: csr_matrix, nbd_states: np.ndarray
) -> csr_matrix:
    """
    Constructs a graph from a neighborhood matrix (see ``find_nbd``):

    - Expands matrix to make it square. Rows are rearranged to align them
        with columns, indexed by ``nbd_states``. I.e. row ``i`` corresponds to
        ``nbd_states[i]``.

    - Symmetrizes the matrix.

    - Converts to a graph by thresholding at 0.
    """
    # Get the non-zero elements of the nbd_matrix.
    nonzero_row, nonzero_col = nbd_matrix.nonzero()

    # Map the original indices of the states to the corresponding indices in nbd_states.
    state_indices = np.searchsorted(nbd_states, states)
    mapped_row_indices = state_indices[nonzero_row]

    # Create two COO matrices: one for the original non-zero elements, and one for the transposed elements.
    data = np.ones_like(nonzero_row, dtype=np.uint8)
    coo_mat = coo_matrix(
        (data, (mapped_row_indices, nonzero_col)), shape=(len(nbd_states), nbd_matrix.shape[1])
    )
    coo_mat_transpose = coo_matrix(
        (data, (nonzero_col, mapped_row_indices)), shape=(nbd_matrix.shape[1], len(nbd_states))
    )

    # Add the two COO matrices and convert to a CSR matrix for efficient arithmetic operations.
    symmetric_matrix = (coo_mat + coo_mat_transpose).tocsr()

    # Threshold at 0.
    return (symmetric_matrix != 0).astype(np.uint8)

In [7]:
def true_relsigns(system: SpinSystem) -> Callable[[npt.NDArray], npt.NDArray]:
    def relings(cluster: npt.NDArray) -> npt.NDArray:
        return np.sign(system.get_ground_state_coeffs(cluster)) * np.random.choice([-1, 1])

    return relings


def almost_true_relsigns(system: SpinSystem, eps: float) -> Callable[[npt.NDArray], npt.NDArray]:
    def relings(cluster: npt.NDArray) -> npt.NDArray:
        return np.sign(system.get_ground_state_coeffs(cluster)) * np.random.choice(
            [1, -1], p=[1 - eps, eps], size=len(cluster)
        )

    return relings

In [8]:
def transfer_signs_to_H(
    states: npt.NDArray,
    M: csr_matrix,
    nbd_states: npt.NDArray,
    relsign_fn: Callable[[npt.NDArray], npt.NDArray],
):
    """
    Moves the signs from the relative signs to the Hamiltonian matrix.
    """
    graph = nbd_matrix_to_graph(states, M, nbd_states)
    _, labels = connected_components(graph, directed=False)
    relsigns = np.empty(len(nbd_states), dtype=np.int8)
    for component in np.unique(labels):
        component_mask = labels == component
        cluster = nbd_states[component_mask]
        relsigns[component_mask] = relsign_fn(cluster)

    state_indices = np.searchsorted(nbd_states, states)

    return diags(relsigns[state_indices], format="csr") @ M @ diags(relsigns, format="csr")

In [9]:
def safe_exp_numpy(x: npt.NDArray, normalise: bool = True) -> npt.NDArray:
    r"""Calculate ``exp(x)`` avoiding overflows. Result is not equal to
    ``exp(x)``, but rather proportional to it. If ``normalise==True``, then
    this function makes sure that output tensor elements sum up to 1.
    """
    x = x - x.max()
    np.exp(x, out=x)
    if normalise:
        x /= x.sum()
    return x

In [10]:
def compute_local_energies(
    hamiltonian: ls.Operator,
    states: npt.NDArray[np.uint64],
    relsigns_fn: Callable[[npt.NDArray[np.uint64]], npt.NDArray[np.int8]],
    log_prob_fn: Callable[[npt.NDArray[np.uint64]], npt.NDArray[np.float64]],
) -> npt.NDArray[np.float64]:
    nbd_matrix, nbd_states = find_nbd(hamiltonian, states)
    nbd_matrix_w_signs = transfer_signs_to_H(states, nbd_matrix, nbd_states, relsigns_fn)
    abs_psi_nbd = safe_exp_numpy(log_prob_fn(nbd_states) * 0.5)
    states_indices = np.searchsorted(nbd_states, states)
    abs_psi_states = abs_psi_nbd[states_indices]
    return nbd_matrix_w_signs @ abs_psi_nbd / abs_psi_states

In [11]:
def test_full_energy():
    lattice = ChainLattice(10)
    system = HeisenbergJ1J2(lattice, J1=1, ground_state_cache_dir=Path("groundstates"))
    true_energy, ground_state = system.get_eigenstates(1)
    true_energy = true_energy[0]
    ground_state = ground_state[:, 0]

    states = system.canonical_basis.states

    def true_logprob(states: npt.NDArray[np.uint64]) -> npt.NDArray[np.float64]:
        return 2 * np.log(np.abs(system.get_ground_state_coeffs(states))) # type ignore
    
    E_loc = compute_local_energies(
        system.hamiltonian,
        states,
        true_relsigns(system),
        true_logprob,
    )

    assert np.isclose(E_loc @ (ground_state ** 2), true_energy)

test_full_energy()

[Debug]   [LOCALE0]   ls_chpl_enumerate_representatives ...
/tmp/nix-shell.2f46E4/ipykernel_2018/3916729551.py:46: ComplexWarning: Casting complex values to real discards the imaginary part
  coeffs_data = np.concatenate(coeff_rows).astype(np.float64)


In [12]:
def test_local_E_loc_consistency():
    lattice = ChainLattice(10)
    system = HeisenbergJ1J2(lattice, J1=1, ground_state_cache_dir=Path("groundstates"))
    system.get_eigenstates(1)

    def true_logprob(states: npt.NDArray[np.uint64]) -> npt.NDArray[np.float64]:
        return 2 * np.log(np.abs(system.get_ground_state_coeffs(states)))  # type ignore

    E_loc_full = compute_local_energies(
        system.hamiltonian,
        system.canonical_basis.states,
        true_relsigns(system),
        true_logprob,
    )

    for size in range(1, len(system.canonical_basis.states), 10):
        print(f"{size=}", end="...")
        sampled_states = np.random.choice(system.canonical_basis.states, size=size)
        E_loc_sampled = compute_local_energies(
            system.hamiltonian,
            sampled_states,
            true_relsigns(system),
            true_logprob,
        )

        assert np.allclose(
            E_loc_full[np.searchsorted(system.canonical_basis.states, sampled_states)],
            E_loc_sampled,
        )

        print("ok")


test_local_E_loc_consistency()

[Debug]   [LOCALE0]   ls_chpl_enumerate_representatives ...
/tmp/nix-shell.2f46E4/ipykernel_2018/3916729551.py:46: ComplexWarning: Casting complex values to real discards the imaginary part
  coeffs_data = np.concatenate(coeff_rows).astype(np.float64)


size=1...ok
size=11...ok
size=21...ok
size=31...ok
size=41...ok
size=51...ok
size=61...ok
size=71...ok
size=81...ok
size=91...ok
size=101...ok
size=111...ok
size=121...ok
size=131...ok
size=141...

ok
size=151...ok
size=161...ok
size=171...ok
size=181...ok
size=191...ok
size=201...ok
size=211...ok
size=221...ok
size=231...ok
size=241...ok
size=251...ok


In [13]:

# Calculate the gradient norms
# FROM: GPT-4
def get_gradient_norm(parameters):
    grads = []
    for p in parameters:
        if p.grad is not None:
            grads.append(p.grad)
    grad_vector = parameters_to_vector(grads)
    return torch.linalg.norm(grad_vector)

# END FROM

In [26]:
def differentiable_safe_exp(x: Tensor, normalise: bool = True) -> Tensor:
    r"""Calculate ``exp(x)`` avoiding overflows. Result is not equal to
    ``exp(x)``, but rather proportional to it. If ``normalise==True``, then
    this function makes sure that output tensor elements sum up to 1.
    """
    x = x - torch.max(x)
    x = torch.exp(x)
    if normalise:
        x = x / torch.sum(x)
    return x

In [27]:
torch.autograd.set_detect_anomaly(False)

In [56]:
# n_samples = 48620
n_samples = 1000
lr = 1e-2
momentum = 0.0
batch_size = 64
sign_noise = 0.0
weight_decay = 0
annealing_steps = 0 # 400
initial_temp = 3

lattice = KagomeLattice(2, 2)
# lattice = ChainLattice(10)
system = HeisenbergJ1J2(
    lattice=lattice,
    J1=1,
    J2=1,
    ground_state_cache_dir=Path("groundstates"),
    use_symmetries=False,
    spin_inversion=None,
)
true_energy, _ = system.get_eigenstates(1)
true_energy = true_energy[0]

if len(system.canonical_basis.states) > 1000:
    eval_set = np.random.choice(system.canonical_basis.states, 1000, replace=False)
else:
    eval_set = system.canonical_basis.states

log_prob_fn = LogProbDenseNet(system, n_hidden=128, hidden_layers=2)

# log_prob_fn = KagomeCNNRegression(system.lattice, hidden_channels1=32, hidden_channels2=64)
optimizer = torch.optim.SGD(log_prob_fn.parameters(), lr=lr, momentum=momentum)
# optimizer = torch.optim.Adam(log_prob_fn.parameters(), lr=lr, weight_decay=weight_decay)

true_amplitudes = torch.from_numpy(np.abs(system.get_ground_state_coeffs(eval_set)))
relsigns_fn = almost_true_relsigns(system, eps=sign_noise)

writer = SummaryWriter(
    log_dir=(
        f"experiments/{datetime.now().strftime('%y_%m_%d')}/{datetime.now().strftime('%H_%M_%S')}"
    )
)


for step in range(10000):
    # states, log_probs, all_probs = sample_exactly(
    #     log_prob_fn,
    #     system.basis,
    #     SamplingOptions(
    #         number_samples=n_samples,
    #         number_chains=1,
    #         mode="exact",
    #         sweep_size=1,
    #         number_discarded=0,
    #     ),
    #     return_all_probs=True
    # )
    states, log_probs, _extra = sample_full(
        log_prob_fn,
        system.basis,
        SamplingOptions(
            number_samples=1,
            number_chains=1,
            mode="full",
            sweep_size=1,
            number_discarded=0,
        ),
    )  

    states = states.view(-1)
    weights = _extra["weights"].view(-1)
    all_probs = weights


    ipr = torch.sum(all_probs**2)
    writer.add_scalar("loss/ipr", ipr, step)

    # states, weights = torch.unique(states.view(-1), return_counts=True)
    # weights = weights.float() / torch.sum(weights)
    E = compute_local_energies(
        system.hamiltonian,
        states.detach().numpy(),
        relsigns_fn=relsigns_fn,
        log_prob_fn=lambda s: log_prob_fn(torch.from_numpy(s)).view(-1).detach().numpy(),
    )
    E = torch.from_numpy(E).to(torch.float32)

    # states = states.view(-1, states.size(-1))
    # log_probs = log_probs.view(-1)
    # weights = weights.view(-1)

    # Compute output gradient
    with torch.no_grad():
        grad = 4 * (E - E @ weights) * weights
        # coeff 4 is due to: 2 from formula, 2 due to we are working with log probs
        # instead of log amplitudes

        grad = grad.view(-1, 1)
        grad_norm = torch.linalg.norm(grad)
        #    logger.info("‖∇E‖₂ = {}", grad_norm)
        writer.add_scalar("loss/‖∇E‖₂", grad_norm, step)
        writer.add_scalar("loss/E_variance", grad_norm / n_samples, step)

        # Calculate full energy
        # E_full = E @ safe_exp(log_prob_fn(states).view(-1), normalise=True)
        # writer.add_scalar("loss/E_full", E_full - torch.tensor(true_energy), step)

    optimizer.zero_grad()
    # batch_size = self.config.inference_batch_size

    # Computing gradients for the amplitude network
    # logger.info("Computing gradients...")
    # if _should_optimize(self.config.amplitude):
    #     self.config.amplitude.train()
    forward_fn = log_prob_fn
    for states_chunk, grad_chunk in split_into_batches((states.view(-1, 1), grad), batch_size):
        output = forward_fn(states_chunk.view(-1))
        output.backward(grad_chunk, retain_graph=True)

        if step < annealing_steps:
            temp = initial_temp * (1 - step / annealing_steps)
            probs = differentiable_safe_exp(output, normalise=True)
            entropy = -torch.sum(probs * torch.log(probs))
            entropy_loss = -temp * entropy
            entropy_loss.backward()

    full_gradient_norm = get_gradient_norm(forward_fn.parameters())
    writer.add_scalar("loss/full_gradient_norm", full_gradient_norm, step)

    optimizer.step()

    predicted_amplitudes = safe_exp(log_prob_fn(eval_set) * 0.5)

    overlap = find_overlap(true_amplitudes, predicted_amplitudes)
    writer.add_scalar("overlap", overlap, step)
    logger.info(
        f"{step}: overlap = {overlap:.3f}, ‖∇E‖₂ = {grad_norm:.3f}, full_gradient_norm = {full_gradient_norm:.3f}"
    )

[Debug]   [LOCALE0]   ls_chpl_enumerate_representatives ...
/tmp/nix-shell.2f46E4/ipykernel_2018/3916729551.py:46: ComplexWarning: Casting complex values to real discards the imaginary part
  coeffs_data = np.concatenate(coeff_rows).astype(np.float64)
2023-08-01 21:36:46.622 | INFO     | __main__:<module>:136 - 0: overlap = 0.615, ‖∇E‖₂ = 1.364, full_gradient_norm = 2.436
2023-08-01 21:36:46.882 | INFO     | __main__:<module>:136 - 1: overlap = 0.615, ‖∇E‖₂ = 1.364, full_gradient_norm = 2.377
2023-08-01 21:36:47.135 | INFO     | __main__:<module>:136 - 2: overlap = 0.616, ‖∇E‖₂ = 1.364, full_gradient_norm = 2.360
2023-08-01 21:36:47.392 | INFO     | __main__:<module>:136 - 3: overlap = 0.616, ‖∇E‖₂ = 1.364, full_gradient_norm = 2.332
2023-08-01 21:36:47.645 | INFO     | __main__:<module>:136 - 4: overlap = 0.616, ‖∇E‖₂ = 1.364, full_gradient_norm = 2.311
2023-08-01 21:36:47.905 | INFO     | __main__:<module>:136 - 5: overlap = 0.617, ‖∇E‖₂ = 1.364, full_gradient_norm = 2.316
2023-08-01

KeyboardInterrupt: 

In [51]:
system.canonical_basis.states.shape

(924,)

In [57]:
(system.get_ground_state_in_canonical_basis() ** 4).sum()

0.005998460942365182